In [1]:
import os
os.chdir('../')

In [2]:
%pwd

'/Users/kiranprasadjp/Documents/Pros/NeuronWireTracingEngine'

In [3]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class TestConfig:
    root_dir: Path
    test_pth: Path
    model_pth: Path
    val_images_dir: Path
    val_labels_dir: Path

In [4]:
from src.neuronTracer import *
from src.neuronTracer.constants import *
from src.neuronTracer.utils.common import read_yaml, create_directories

In [5]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])


    
    def get_test_config(self) -> TestConfig:
        config = self.config.model_test

        create_directories([config.root_dir])

        test_config = TestConfig(
            root_dir= Path(config.root_dir),
            test_pth= Path(config.test_pth),
            model_pth= Path(config.model_pth),
            val_images_dir= Path(config.val_images_dir),
            val_labels_dir= Path(config.val_labels_dir)
            
        )

        return test_config

In [6]:
import torch
import torch.nn as nn
import numpy as np
import zarr
import tifffile
import os
import torch.nn.functional as F
from tqdm import tqdm

In [7]:

# ─────────────────────────────────────────────────────────────────────────────
# 1. Rebuild the Advanced Architecture (Matches your training script perfectly)
# ─────────────────────────────────────────────────────────────────────────────

class DinoAdapter(nn.Module):
    def __init__(self):
        super().__init__()
        self.resize = nn.Upsample(size=(224, 224), mode='bilinear', align_corners=False)

    def forward(self, x):
        x = self.resize(x)         # [B, 1, 224, 224]
        x = x.repeat(1, 3, 1, 1)   # [B, 3, 224, 224]
        return x

class BotanistDecoder(nn.Module):
    PATCH_GRID = 16   # 224 // 14 = 16 patches per side

    def __init__(self):
        super().__init__()
        self.dino_dim = 768

        self.deconv1 = nn.Sequential(
            nn.ConvTranspose2d(768, 256, kernel_size=4, stride=4),   # 16 → 64
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
        )
        self.deconv2 = nn.Sequential(
            nn.ConvTranspose2d(256, 64, kernel_size=2, stride=2),    # 64 → 128
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
        )
        self.deconv3 = nn.Sequential(
            nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2),     # 128 → 256
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
        )
        self.final = nn.Sequential(
            nn.Conv2d(32, 1, kernel_size=1),
            nn.Upsample(size=(224, 224), mode='bilinear', align_corners=False),
            # NO Sigmoid here! We output raw logits.
        )

    def forward(self, patch_tokens):
        # patch_tokens: [B, 256, 768]
        B = patch_tokens.shape[0]
        x = patch_tokens.transpose(1, 2).reshape(B, self.dino_dim, self.PATCH_GRID, self.PATCH_GRID)
        x = self.deconv1(x)   # [B, 256,  64,  64]
        x = self.deconv2(x)   # [B,  64, 128, 128]
        x = self.deconv3(x)   # [B,  32, 256, 256]
        return self.final(x)  # [B,   1, 224, 224] raw logits

class HybridDinoTracker(nn.Module):
    def __init__(self):
        super().__init__()
        self.adapter = DinoAdapter()
        logger.info("Waking up Meta's DINOv2 Satellite...")
        self.dino_encoder = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14', trust_repo=True)
        
        # We don't need to freeze here since we use torch.no_grad() in inference anyway
        self.botanist_decoder = BotanistDecoder()

    def forward(self, x):
        x = self.adapter(x)
        # Pull the spatial patch grid instead of the flat CLS token
        with torch.no_grad():
            features = self.dino_encoder.get_intermediate_layers(x, n=1)
        patch_tokens = features[0]
        return self.botanist_decoder(patch_tokens)

# ─────────────────────────────────────────────────────────────────────────────
# 2. The Test Taker
# ─────────────────────────────────────────────────────────────────────────────

class Test:
    def __init__(self, config: TestConfig):
        self.config=config
        self.device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
        self.model = HybridDinoTracker().to(self.device)
        self.weights_path = self.config.model_pth

    def take_final(self):
        logger.info("Preparing for the Final Exam on Apple Silicon...")
        
        
        if os.path.exists(self.weights_path):
            self.model.botanist_decoder.load_state_dict(torch.load(self.weights_path, map_location=self.device, weights_only=True))
            logger.info("✅ Successfully loaded your highly trained custom brain!")
        else:
            logger.info(f"❌ ERROR: Could not find {self.weights_path}.")
            return
            
        self.model.eval()
        
        logger.info("Opening the blank test dataset...")
        test_zarr = zarr.open(self.config.test_pth, mode='r')
        
        final_3d_canvas = np.zeros(test_zarr.shape, dtype=np.float32)
        Z_depth = test_zarr.shape[0]
        box_size = 64 
        
        logger.info("The AI is now tracing the final exam. Please wait...")
        
        with torch.no_grad(): 
            for z in range(Z_depth):
                for y in range(0, test_zarr.shape[1], box_size):
                    for x in range(0, test_zarr.shape[2], box_size):
                        
                        test_chunk = test_zarr[z, y:y+box_size, x:x+box_size]
                        
                        if test_chunk.shape[0] != box_size or test_chunk.shape[1] != box_size:
                            continue
                            
                        input_tensor = torch.from_numpy(test_chunk).float().unsqueeze(0).unsqueeze(0).to(self.device)
                        
                        # 1. The AI draws the lines (Outputs 224x224 raw logits)
                        raw_logits = self.model(input_tensor)
                        
                        # 2. Convert to percentages
                        ai_tracing_224 = torch.sigmoid(raw_logits)
                        
                        # 🔥 THE FIX: Shrink the 224x224 drawing back down to 64x64
                        ai_tracing_64 = F.interpolate(
                            ai_tracing_224, 
                            size=(box_size, box_size), 
                            mode='bilinear', 
                            align_corners=False
                        )
                        
                        # 3. Paste the properly sized drawing onto our massive billboard
                        final_3d_canvas[z, y:y+box_size, x:x+box_size] = ai_tracing_64.squeeze().cpu().numpy()
                        
                if z % 10 == 0:
                    logger.info(f"Traced slice {z} / {Z_depth}...")

        logger.info("🎉 Exam finished! Saving the completed 3D tracing...")
        output_path = os.path.join(self.config.root_dir, "final.tif")
        tifffile.imwrite(output_path, final_3d_canvas)
        logger.info("SUCCESS! You can now upload 'final.tif' to the SNEMI3D Leaderboard.")

    def run_official_evaluation(self):
        logger.info("Setting up the Validation Grader on Apple Silicon...")
        
        
        if os.path.exists(self.weights_path):
            self.model.botanist_decoder.load_state_dict(torch.load(self.weights_path, map_location=self.device, weights_only=True))
            logger.info("✅ Botanist brain loaded successfully!")
        else:
            logger.info(f"❌ Could not find {self.weights_path}")
            return
            
        self.model.eval()
        
        # 2. Point to the NEW validation folders
        val_images_dir = self.config.val_images_dir
        val_labels_dir = self.config.val_labels_dir
        
        val_files = sorted([f for f in os.listdir(val_images_dir) if f.endswith('.pt')])
        logger.info(f"Found {len(val_files)} isolated validation files. Beginning grading...")
        
        # Trackers for our biological metrics
        total_tp = 0
        total_fp = 0
        total_fn = 0
        
        with torch.no_grad():
            for file in tqdm(val_files, desc="Grading AI"):
                # Load the raw .pt files directly into memory
                img_tensor = torch.load(os.path.join(val_images_dir, file), map_location=self.device, weights_only=True)
                lbl_tensor = torch.load(os.path.join(val_labels_dir, file), map_location='cpu', weights_only=True).numpy()
                
                # 🔥 THE 21-CHANNEL FIX: Flip [1, 7, 64, 64] to [7, 1, 64, 64]
                img_tensor = img_tensor.transpose(0, 1)
                
                # 1. AI makes a prediction on all 7 slices simultaneously
                raw_logits = self.model(img_tensor)
                ai_tracing_224 = torch.sigmoid(raw_logits)
                
                # 2. Shrink the drawing back to 64x64 to match the answer key
                original_size = lbl_tensor.shape[-1] 
                ai_tracing_resized = F.interpolate(
                    ai_tracing_224, 
                    size=(original_size, original_size), 
                    mode='bilinear', 
                    align_corners=False
                )
                
                # 🔥 FLIP IT BACK: [7, 1, 64, 64] back to [1, 7, 64, 64] to match the label shape
                ai_tracing_final = ai_tracing_resized.transpose(0, 1).cpu().numpy()
                
                # 3. Convert to Hard Yes/No decisions (Threshold at 50% confidence)
                preds_binary = (ai_tracing_final > 0.5).astype(bool)
                truth_binary = (lbl_tensor > 0).astype(bool)
                
                # 4. Calculate Confusion Matrix Metrics on the fly using high-speed NumPy
                total_tp += np.sum(preds_binary & truth_binary)
                total_fp += np.sum(preds_binary & ~truth_binary)
                total_fn += np.sum(~preds_binary & truth_binary)
                
        # Calculate Final Mathematics
        precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0.0
        recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0.0
        dice_score = (2 * total_tp) / (2 * total_tp + total_fp + total_fn) if (2 * total_tp + total_fp + total_fn) > 0 else 0.0
        iou = total_tp / (total_tp + total_fp + total_fn) if (total_tp + total_fp + total_fn) > 0 else 0.0
        
        logger.info("\n" + "="*40)
        logger.info("📊 UNBIASED VALIDATION REPORT")
        logger.info("="*40)
        logger.info(f"Precision (Hallucination Check):  {precision * 100:.2f}%")
        logger.info(f"Recall    (Blind-Spot Check):     {recall * 100:.2f}%")
        logger.info(f"Dice / F1 (Official Accuracy):    {dice_score * 100:.2f}%")
        logger.info(f"IoU       (Geometric Overlap):    {iou * 100:.2f}%")
        logger.info("="*40)


In [8]:
try:
    config = ConfigurationManager()
    test_config = config.get_test_config()
    test = Test(config=test_config)
    test.take_final()
    test.run_official_evaluation()
except Exception as e:
    raise e

[2026-03-24 16:44:43,314: INFO: common: yaml file: config/config.yaml loaded successfully]
[2026-03-24 16:44:43,317: INFO: common: yaml file: params.yaml loaded successfully]
[2026-03-24 16:44:43,318: INFO: common: created directory at: artifacts]
[2026-03-24 16:44:43,318: INFO: common: created directory at: artifacts/test]
[2026-03-24 16:44:43,388: INFO: 1848212189: Waking up Meta's DINOv2 Satellite...]
[2026-03-24 16:44:44,096: INFO: vision_transformer: using MLP layer as FFN]


Using cache found in /Users/kiranprasadjp/.cache/torch/hub/facebookresearch_dinov2_main
/Users/kiranprasadjp/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/Users/kiranprasadjp/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/Users/kiranprasadjp/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


[2026-03-24 16:44:45,496: INFO: 1848212189: Preparing for the Final Exam on Apple Silicon...]
[2026-03-24 16:44:45,772: INFO: 1848212189: ✅ Successfully loaded your highly trained custom brain!]
[2026-03-24 16:44:45,773: INFO: 1848212189: Opening the blank test dataset...]
[2026-03-24 16:44:45,778: INFO: 1848212189: The AI is now tracing the final exam. Please wait...]
[2026-03-24 16:44:53,905: INFO: 1848212189: Traced slice 0 / 100...]
[2026-03-24 16:46:09,705: INFO: 1848212189: Traced slice 10 / 100...]
[2026-03-24 16:47:26,585: INFO: 1848212189: Traced slice 20 / 100...]
[2026-03-24 16:48:41,650: INFO: 1848212189: Traced slice 30 / 100...]
[2026-03-24 16:49:56,953: INFO: 1848212189: Traced slice 40 / 100...]
[2026-03-24 16:51:13,674: INFO: 1848212189: Traced slice 50 / 100...]
[2026-03-24 16:52:35,380: INFO: 1848212189: Traced slice 60 / 100...]
[2026-03-24 16:53:52,998: INFO: 1848212189: Traced slice 70 / 100...]
[2026-03-24 16:55:10,970: INFO: 1848212189: Traced slice 80 / 100...]

Grading AI: 100%|██████████| 3584/3584 [09:42<00:00,  6.15it/s]

[2026-03-24 17:07:23,292: INFO: 1848212189: 
========================================]
[2026-03-24 17:07:23,293: INFO: 1848212189: 📊 UNBIASED VALIDATION REPORT]
[2026-03-24 17:07:23,293: INFO: 1848212189: ========================================]
[2026-03-24 17:07:23,294: INFO: 1848212189: Precision (Hallucination Check):  80.94%]
[2026-03-24 17:07:23,294: INFO: 1848212189: Recall    (Blind-Spot Check):     40.53%]
[2026-03-24 17:07:23,294: INFO: 1848212189: Dice / F1 (Official Accuracy):    54.01%]
[2026-03-24 17:07:23,295: INFO: 1848212189: IoU       (Geometric Overlap):    37.00%]
[2026-03-24 17:07:23,295: INFO: 1848212189: ========================================]


In [2]:
import torch
import torch.nn as nn
import numpy as np
import zarr
import tifffile
import os
import torch.nn.functional as F

# ─────────────────────────────────────────────────────────────────────────────
# 1. Rebuild the Advanced Architecture (Matches your training script perfectly)
# ─────────────────────────────────────────────────────────────────────────────

class DinoAdapter(nn.Module):
    def __init__(self):
        super().__init__()
        self.resize = nn.Upsample(size=(224, 224), mode='bilinear', align_corners=False)

    def forward(self, x):
        x = self.resize(x)         # [B, 1, 224, 224]
        x = x.repeat(1, 3, 1, 1)   # [B, 3, 224, 224]
        return x

class BotanistDecoder(nn.Module):
    PATCH_GRID = 16   # 224 // 14 = 16 patches per side

    def __init__(self):
        super().__init__()
        self.dino_dim = 768

        self.deconv1 = nn.Sequential(
            nn.ConvTranspose2d(768, 256, kernel_size=4, stride=4),   # 16 → 64
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
        )
        self.deconv2 = nn.Sequential(
            nn.ConvTranspose2d(256, 64, kernel_size=2, stride=2),    # 64 → 128
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
        )
        self.deconv3 = nn.Sequential(
            nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2),     # 128 → 256
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
        )
        self.final = nn.Sequential(
            nn.Conv2d(32, 1, kernel_size=1),
            nn.Upsample(size=(224, 224), mode='bilinear', align_corners=False),
            # NO Sigmoid here! We output raw logits.
        )

    def forward(self, patch_tokens):
        # patch_tokens: [B, 256, 768]
        B = patch_tokens.shape[0]
        x = patch_tokens.transpose(1, 2).reshape(B, self.dino_dim, self.PATCH_GRID, self.PATCH_GRID)
        x = self.deconv1(x)   # [B, 256,  64,  64]
        x = self.deconv2(x)   # [B,  64, 128, 128]
        x = self.deconv3(x)   # [B,  32, 256, 256]
        return self.final(x)  # [B,   1, 224, 224] raw logits

class HybridDinoTracker(nn.Module):
    def __init__(self):
        super().__init__()
        self.adapter = DinoAdapter()
        print("Waking up Meta's DINOv2 Satellite...")
        self.dino_encoder = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14', trust_repo=True)
        
        # We don't need to freeze here since we use torch.no_grad() in inference anyway
        self.botanist_decoder = BotanistDecoder()

    def forward(self, x):
        x = self.adapter(x)
        # Pull the spatial patch grid instead of the flat CLS token
        with torch.no_grad():
            features = self.dino_encoder.get_intermediate_layers(x, n=1)
        patch_tokens = features[0]
        return self.botanist_decoder(patch_tokens)

# ─────────────────────────────────────────────────────────────────────────────
# 2. The Test Taker
# ─────────────────────────────────────────────────────────────────────────────


def take_final_exam():
    print("Preparing for the Final Exam on Apple Silicon...")
    device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
    
    model = HybridDinoTracker().to(device)
    
    weights_path = "research/my_dino_decoder (1).pth"
    if os.path.exists(weights_path):
        model.botanist_decoder.load_state_dict(torch.load(weights_path, map_location=device, weights_only=True))
        print("✅ Successfully loaded your highly trained custom brain!")
    else:
        print(f"❌ ERROR: Could not find {weights_path}.")
        return
        
    model.eval()
    
    print("Opening the blank test dataset...")
    test_zarr = zarr.open("artifacts/data_transform/test.zarr", mode='r')
    
    final_3d_canvas = np.zeros(test_zarr.shape, dtype=np.float32)
    Z_depth = test_zarr.shape[0]
    box_size = 64 
    
    print("The AI is now tracing the final exam. Please wait...")
    
    with torch.no_grad(): 
        for z in range(Z_depth):
            for y in range(0, test_zarr.shape[1], box_size):
                for x in range(0, test_zarr.shape[2], box_size):
                    
                    test_chunk = test_zarr[z, y:y+box_size, x:x+box_size]
                    
                    if test_chunk.shape[0] != box_size or test_chunk.shape[1] != box_size:
                        continue
                        
                    input_tensor = torch.from_numpy(test_chunk).float().unsqueeze(0).unsqueeze(0).to(device)
                    
                    # 1. The AI draws the lines (Outputs 224x224 raw logits)
                    raw_logits = model(input_tensor)
                    
                    # 2. Convert to percentages
                    ai_tracing_224 = torch.sigmoid(raw_logits)
                    
                    # 🔥 THE FIX: Shrink the 224x224 drawing back down to 64x64
                    ai_tracing_64 = F.interpolate(
                        ai_tracing_224, 
                        size=(box_size, box_size), 
                        mode='bilinear', 
                        align_corners=False
                    )
                    
                    # 3. Paste the properly sized drawing onto our massive billboard
                    final_3d_canvas[z, y:y+box_size, x:x+box_size] = ai_tracing_64.squeeze().cpu().numpy()
                    
            if z % 10 == 0:
                print(f"Traced slice {z} / {Z_depth}...")

    print("🎉 Exam finished! Saving the completed 3D tracing...")
    tifffile.imwrite("final_submission.tif", final_3d_canvas)
    print("SUCCESS! You can now upload 'final_submission.tif' to the SNEMI3D Leaderboard.")
if __name__ == "__main__":
    take_final_exam()

Preparing for the Final Exam on Apple Silicon...
Waking up Meta's DINOv2 Satellite...


Using cache found in /Users/kiranprasadjp/.cache/torch/hub/facebookresearch_dinov2_main
/Users/kiranprasadjp/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/Users/kiranprasadjp/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/Users/kiranprasadjp/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


✅ Successfully loaded your highly trained custom brain!
Opening the blank test dataset...
The AI is now tracing the final exam. Please wait...
Traced slice 0 / 100...
Traced slice 10 / 100...
Traced slice 20 / 100...
Traced slice 30 / 100...
Traced slice 40 / 100...
Traced slice 50 / 100...
Traced slice 60 / 100...
Traced slice 70 / 100...
Traced slice 80 / 100...
Traced slice 90 / 100...
🎉 Exam finished! Saving the completed 3D tracing...
SUCCESS! You can now upload 'final_submission.tif' to the SNEMI3D Leaderboard.


In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import os
from tqdm import tqdm

# ─────────────────────────────────────────────────────────────────────────────
# 1. The Architecture Blueprint (Matches your trained brain)
# ─────────────────────────────────────────────────────────────────────────────
class DinoAdapter(nn.Module):
    def __init__(self):
        super().__init__()
        self.resize = nn.Upsample(size=(224, 224), mode='bilinear', align_corners=False)
    def forward(self, x):
        return self.resize(x).repeat(1, 3, 1, 1)

class BotanistDecoder(nn.Module):
    PATCH_GRID = 16 
    def __init__(self):
        super().__init__()
        self.dino_dim = 768
        self.deconv1 = nn.Sequential(nn.ConvTranspose2d(768, 256, 4, 4), nn.BatchNorm2d(256), nn.ReLU(True))
        self.deconv2 = nn.Sequential(nn.ConvTranspose2d(256, 64, 2, 2), nn.BatchNorm2d(64), nn.ReLU(True))
        self.deconv3 = nn.Sequential(nn.ConvTranspose2d(64, 32, 2, 2), nn.BatchNorm2d(32), nn.ReLU(True))
        self.final = nn.Sequential(
            nn.Conv2d(32, 1, 1),
            nn.Upsample(size=(224, 224), mode='bilinear', align_corners=False)
        )
    def forward(self, patch_tokens):
        B = patch_tokens.shape[0]
        x = patch_tokens.transpose(1, 2).reshape(B, self.dino_dim, self.PATCH_GRID, self.PATCH_GRID)
        x = self.deconv1(x)
        x = self.deconv2(x)
        x = self.deconv3(x)
        return self.final(x) 

class HybridDinoTracker(nn.Module):
    def __init__(self):
        super().__init__()
        self.adapter = DinoAdapter()
        self.dino_encoder = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14', trust_repo=True)
        self.botanist_decoder = BotanistDecoder()
    def forward(self, x):
        x = self.adapter(x)
        with torch.no_grad():
            features = self.dino_encoder.get_intermediate_layers(x, n=1)
        return self.botanist_decoder(features[0])

# ─────────────────────────────────────────────────────────────────────────────
# 2. The High-Speed Grader
# ─────────────────────────────────────────────────────────────────────────────
def run_official_evaluation():
    print("Setting up the Validation Grader on Apple Silicon...")
    device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
    
    # 1. Load the Model
    model = HybridDinoTracker().to(device)
    weights_path = "research/my_dino_decoder (1).pth"
    
    if os.path.exists(weights_path):
        model.botanist_decoder.load_state_dict(torch.load(weights_path, map_location=device, weights_only=True))
        print("✅ Botanist brain loaded successfully!")
    else:
        print(f"❌ Could not find {weights_path}")
        return
        
    model.eval()
    
    # 2. Point to the NEW validation folders
    val_images_dir = "artifacts/torch_convert/images"
    val_labels_dir = "artifacts/torch_convert/labels"
    
    val_files = sorted([f for f in os.listdir(val_images_dir) if f.endswith('.pt')])
    print(f"Found {len(val_files)} isolated validation files. Beginning grading...")
    
    # Trackers for our biological metrics
    total_tp = 0
    total_fp = 0
    total_fn = 0
    
    with torch.no_grad():
        for file in tqdm(val_files, desc="Grading AI"):
            # Load the raw .pt files directly into memory
            img_tensor = torch.load(os.path.join(val_images_dir, file), map_location=device, weights_only=True)
            lbl_tensor = torch.load(os.path.join(val_labels_dir, file), map_location='cpu', weights_only=True).numpy()
            
            # 🔥 THE 21-CHANNEL FIX: Flip [1, 7, 64, 64] to [7, 1, 64, 64]
            img_tensor = img_tensor.transpose(0, 1)
            
            # 1. AI makes a prediction on all 7 slices simultaneously
            raw_logits = model(img_tensor)
            ai_tracing_224 = torch.sigmoid(raw_logits)
            
            # 2. Shrink the drawing back to 64x64 to match the answer key
            original_size = lbl_tensor.shape[-1] 
            ai_tracing_resized = F.interpolate(
                ai_tracing_224, 
                size=(original_size, original_size), 
                mode='bilinear', 
                align_corners=False
            )
            
            # 🔥 FLIP IT BACK: [7, 1, 64, 64] back to [1, 7, 64, 64] to match the label shape
            ai_tracing_final = ai_tracing_resized.transpose(0, 1).cpu().numpy()
            
            # 3. Convert to Hard Yes/No decisions (Threshold at 50% confidence)
            preds_binary = (ai_tracing_final > 0.5).astype(bool)
            truth_binary = (lbl_tensor > 0).astype(bool)
            
            # 4. Calculate Confusion Matrix Metrics on the fly using high-speed NumPy
            total_tp += np.sum(preds_binary & truth_binary)
            total_fp += np.sum(preds_binary & ~truth_binary)
            total_fn += np.sum(~preds_binary & truth_binary)
            
    # Calculate Final Mathematics
    precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0.0
    recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0.0
    dice_score = (2 * total_tp) / (2 * total_tp + total_fp + total_fn) if (2 * total_tp + total_fp + total_fn) > 0 else 0.0
    iou = total_tp / (total_tp + total_fp + total_fn) if (total_tp + total_fp + total_fn) > 0 else 0.0
    
    print("\n" + "="*40)
    print("📊 UNBIASED VALIDATION REPORT")
    print("="*40)
    print(f"Precision (Hallucination Check):  {precision * 100:.2f}%")
    print(f"Recall    (Blind-Spot Check):     {recall * 100:.2f}%")
    print(f"Dice / F1 (Official Accuracy):    {dice_score * 100:.2f}%")
    print(f"IoU       (Geometric Overlap):    {iou * 100:.2f}%")
    print("="*40)

if __name__ == "__main__":
    run_official_evaluation()

Setting up the Validation Grader on Apple Silicon...


Using cache found in /Users/kiranprasadjp/.cache/torch/hub/facebookresearch_dinov2_main


✅ Botanist brain loaded successfully!
Found 3584 isolated validation files. Beginning grading...


Grading AI: 100%|██████████| 3584/3584 [09:10<00:00,  6.51it/s]


📊 UNBIASED VALIDATION REPORT
Precision (Hallucination Check):  80.94%
Recall    (Blind-Spot Check):     40.53%
Dice / F1 (Official Accuracy):    54.01%
IoU       (Geometric Overlap):    37.00%
